In [1]:
import os
import json
import time
from rdflib import ConjunctiveGraph, URIRef, Literal
from rdflib.namespace import RDF

# loading datasets from dump file to a json file
INPUT_FOLDER = "C:/Users/deper/Documents/data gov uk"
OUTPUT_FILE = "gov_data_uk.json"

BATCH_SIZE = 22   # number of files per batch
MAX_FILES = None
# RDF URIs
DCAT_DATASET = URIRef("http://www.w3.org/ns/dcat#Dataset")
DCT_TITLE = URIRef("http://purl.org/dc/terms/title")
DCT_DESCRIPTION = URIRef("http://purl.org/dc/terms/description")
DCAT_KEYWORD = URIRef("http://www.w3.org/ns/dcat#keyword")

# -------- GET FILE LIST --------
trig_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".trig")
]

trig_files = sorted(trig_files)

if MAX_FILES:
    trig_files = trig_files[:MAX_FILES]

print(f"Processing {len(trig_files)} files in batches of {BATCH_SIZE}\n")

# -------- INIT OUTPUT --------
f_out = open(OUTPUT_FILE, "w", encoding="utf-8")
f_out.write("[\n")

first = True
total_datasets = 0
skipped_no_keywords = 0

# -------- PROCESS IN BATCHES --------
for i in range(0, len(trig_files), BATCH_SIZE):

    batch = trig_files[i:i+BATCH_SIZE]

    print(f"\n[LOAD BATCH] Files {i+1} → {i+len(batch)}")

    g = ConjunctiveGraph()
    batch_start = time.time()

    # -------- LOAD FILES --------
    for filepath in batch:
        filename = os.path.basename(filepath)
        print(f"[LOAD ] {filename}")

        try:
            g.parse(filepath, format="trig")
        except Exception as e:
            print(f"[ERROR] {filename}: {e}")

    print(f"[INFO ] Batch loaded in {time.time() - batch_start:.2f}s")

    # -------- EXTRACT DATASETS --------
    extract_start = time.time()

    datasets = set(s for s, _, _ in g.triples((None, RDF.type, DCAT_DATASET)))
    print(f"[INFO ] Found {len(datasets)} datasets in batch")

    dataset_data = {}

    # -------- SINGLE PASS --------
    for s, p, o in g:
        if s not in datasets:
            continue

        if s not in dataset_data:
            dataset_data[s] = {
                "title": [],
                "description": [],
                "keywords": set()
            }

        if p == DCT_TITLE and isinstance(o, Literal) and o.language == "en":
            dataset_data[s]["title"].append(str(o))

        elif p == DCT_DESCRIPTION and isinstance(o, Literal) and o.language == "en":
            dataset_data[s]["description"].append(str(o))

        elif p == DCAT_KEYWORD:
            dataset_data[s]["keywords"].add(str(o))

    # -------- WRITE RESULTS --------
    batch_count = 0

    for ds, data in dataset_data.items():
        # ✅ require: English title + at least one keyword
        if not data["title"] or not data["keywords"]:
            if not data["keywords"]:
                skipped_no_keywords += 1
            continue

        record = {
            "dataset_id": str(ds),
            "title": data["title"][0],
            "description": data["description"][0] if data["description"] else None,
            "keywords": sorted(list(data["keywords"]))
        }

        if not first:
            f_out.write(",\n")

        f_out.write(json.dumps(record, ensure_ascii=False))
        first = False

        batch_count += 1
        total_datasets += 1

    print(
        f"[DONE ] Batch processed: {batch_count} datasets "
        f"({time.time() - extract_start:.2f}s)"
    )
    print(f"[INFO ] Skipped (no keywords): {skipped_no_keywords}")

# -------- CLOSE FILE --------
f_out.write("\n]\n")
f_out.close()

print(f"\nDone. Extracted {total_datasets} datasets.")
print(f"Total skipped (no keywords): {skipped_no_keywords}")

Processing 176 files in batches of 22


[LOAD BATCH] Files 1 → 22
[LOAD ] Dataset_014657da-d6ca-44b2-ab32-266d772955ae.trig


C:\Users\deper\AppData\Local\Temp\ipykernel_16784\250664126.py:48: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


[LOAD ] Dataset_01be406f-b3f1-43a2-8547-6f44332812ec.trig
[LOAD ] Dataset_03640eb1-88b0-4b70-b364-bb6117d87705.trig
[LOAD ] Dataset_03d37d1e-232a-45ed-9ab6-ae53ea2c8a48.trig
[LOAD ] Dataset_051b1d0e-5bec-438a-b04b-0ac8c47f8080.trig
[LOAD ] Dataset_060c0cec-3ab1-418d-8013-c7fe542e99fe.trig
[LOAD ] Dataset_06879bbc-24c9-46f0-8913-a8c8edcdea6e.trig
[LOAD ] Dataset_08ebf12a-a3b7-4b5b-9270-3898dedc561a.trig
[LOAD ] Dataset_0a76fa6f-4e27-417c-99f9-27d830955f1c.trig
[LOAD ] Dataset_0a89649a-d7fa-452d-912d-2993c3ac92bd.trig
[LOAD ] Dataset_0e4965a2-936b-4e52-9840-7859cbc60258.trig
[LOAD ] Dataset_10e3ef5b-137a-41c0-b4db-4729c85f7857.trig
[LOAD ] Dataset_115e8a9e-6f11-4a56-b701-1b1e7140778c.trig
[LOAD ] Dataset_11c4a660-6bc0-4f05-b33f-065d55cb2c84.trig
[LOAD ] Dataset_12706368-8423-4a63-9c21-ce114060c376.trig
[LOAD ] Dataset_12eb0e3c-0c73-4b3e-9323-0dc1d634d73d.trig
[LOAD ] Dataset_1416eaec-a610-401b-ac68-3a53ef5769e7.trig
[LOAD ] Dataset_1453178f-9a74-4baa-8303-a8bfd30a279e.trig
[LOAD ] Datase